# The librarian's questions, asked in Python and in SQL

Week 0, Thursday. The college librarian, the morning after Wednesday's function worked:

> "The office has moved the issue log into the library's database, and it grows every week. Can the
> same three numbers come straight out of the database, for any week I ask about, without anyone
> copying rows into a notebook?"

This notebook asks each of the librarian's questions twice, once in the Python Wednesday taught and
once in SQL, on the same rows, and checks that the two answers agree. By the end, each Python move
has a SQL clause beside it.

In [1]:
import pathlib
import sys

here = pathlib.Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

# Thursday's database is the library one you made from the setup sheet, reached as your own user.
kit.WAREHOUSE.clear()
kit.WAREHOUSE.update(dbname="library")

kit.side_by_side(
    kit.ladder(["Wednesday: the log in Python", "Thursday: the same log in Postgres"], lit=1,
               title="Two days, one log", show=False),
    kit.flow(["the table", "a count", "a count per group", "a filtered total", "every week at once"], title="What this notebook does, in order", show=False),
)

The setup cell finds the programme's helper, `scripts/c2kit.py`, and points it at the `library`
database. It needs the server running and the table loaded, which is steps 2 to 4 of the setup sheet.

## The table

The `issues` table holds four weeks. Week 1 is Wednesday's class log, row for row. Fetching those
rows into Python gives a list of dictionaries, which is exactly the shape Wednesday's loops expect.

In [2]:
kit.flow(["the table", "a count", "a count per group", "a filtered total", "every week at once"], lit=0, title="Where this section sits")

In [3]:
kit.sql_table("SELECT * FROM issues WHERE week = 1 ORDER BY issue_id", caption="Week 1, from Postgres")

issue_id,week,student,book,dept,days_kept
1,1,S01,Algebra,Maths,12
2,1,S02,Poetry,Arts,5
3,1,S01,Calculus,Maths,20
4,1,S03,Poetry,Arts,9
5,1,S04,Physics,Science,14
6,1,S02,Algebra,Maths,3
7,1,S05,Chemistry,Science,21
8,1,S03,Calculus,Maths,7


In [4]:
log = kit.sql("SELECT issue_id, student, book, dept, days_kept FROM issues WHERE week = 1 ORDER BY issue_id")
print(len(log), "rows fetched; the first is", log[0])
kit.check("week 1 holds Wednesday's 8 issues", len(log) == 8, f"{len(log)} rows")
kit.check("days_kept arrives from Postgres as a whole number", all(type(r["days_kept"]) is int for r in log))

8 rows fetched; the first is {'issue_id': 1, 'student': 'S01', 'book': 'Algebra', 'dept': 'Maths', 'days_kept': 12}


## A count

In Python, a count is three moves: start at 0, add 1 per record, read the total after the loop. In
SQL, all three moves live inside `COUNT(*)`, and `WHERE week = 1` says which rows count.

In [5]:
kit.flow(["the table", "a count", "a count per group", "a filtered total", "every week at once"], lit=1, title="Where this section sits")

In [6]:
count = 0
for issue in log:
    count = count + 1

by_sql = kit.sql("SELECT COUNT(*) AS issues FROM issues WHERE week = 1")[0]["issues"]
print("Python:", count, "  SQL:", by_sql)
kit.check("the loop and COUNT(*) agree on 8", count == by_sql == 8, f"{count} and {by_sql}")

Python: 8   SQL: 8


## A count per group

The busiest department needs one running total per department. In Python that is a dictionary whose
keys are the departments; in SQL it is `GROUP BY dept`, which forms one group per department and runs
`COUNT(*)` inside each.

In [7]:
kit.flow(["the table", "a count", "a count per group", "a filtered total", "every week at once"], lit=2, title="Where this section sits")

In [8]:
counts = {}
for issue in log:
    dept = issue["dept"]
    if dept not in counts:
        counts[dept] = 0
    counts[dept] = counts[dept] + 1

rows = kit.sql("""
    SELECT dept, COUNT(*) AS issues
    FROM issues
    WHERE week = 1
    GROUP BY dept
    ORDER BY issues DESC, dept
""")
by_sql = {r["dept"]: r["issues"] for r in rows}
print("Python:", counts)
print("SQL:   ", by_sql)
kit.check("the dictionary and GROUP BY agree", counts == by_sql, str(by_sql))
kit.check("Maths borrows most, with 4 issues", rows[0]["dept"] == "Maths" and rows[0]["issues"] == 4)

Python: {'Maths': 4, 'Arts': 2, 'Science': 2}
SQL:    {'Maths': 4, 'Arts': 2, 'Science': 2}


## A filtered total, and a deliberate failure

Days late count only the books kept past the 14-day loan. The Python `if` becomes `WHERE days_kept >
14`, and the running total becomes `SUM(days_kept - 14)`.

First, though, a line that looks right and fails. Python accepts either kind of quote around text;
Postgres gives each kind a different job.

In [9]:
kit.flow(["the table", "a count", "a count per group", "a filtered total", "every week at once"], lit=3, title="Where this section sits")

In [10]:
try:
    kit.sql('SELECT COUNT(*) AS n FROM issues WHERE week = 1 AND dept = "Maths"')
except Exception as error:
    print(type(error).__name__)
    print(str(error).strip())

UndefinedColumn
column "Maths" does not exist
LINE 1: ... COUNT(*) AS n FROM issues WHERE week = 1 AND dept = "Maths"
                                                                ^


Double quotes wrap the name of a column or a table, so Postgres went looking for a column called
Maths and found none. Text belongs in single quotes.

In [11]:
maths = kit.sql("SELECT COUNT(*) AS n FROM issues WHERE week = 1 AND dept = 'Maths'")[0]["n"]
kit.check("with single quotes, Maths has 4 issues in week 1", maths == 4, f"{maths}")

late = 0
for issue in log:
    if issue["days_kept"] > 14:
        late = late + (issue["days_kept"] - 14)

by_sql = kit.sql("SELECT SUM(days_kept - 14) AS late FROM issues WHERE week = 1 AND days_kept > 14")[0]["late"]
print("Python:", late, "  SQL:", by_sql)
kit.check("the loop and SUM agree on 13 days late", late == by_sql == 13, f"{late} and {by_sql}")

Python: 13   SQL: 13


## Every week at once

Wednesday wrapped the loop in a function so next week's log could go through it. SQL answers every
week in one query by grouping on `week`. The Python below runs Wednesday-style functions over each
week in turn, and the check compares them with the two grouped queries.

In [12]:
kit.flow(["the table", "a count", "a count per group", "a filtered total", "every week at once"], lit=4, title="Where this section sits")

In [13]:
def issues_in(rows):
    return len(rows)


def late_days(rows):
    total = 0
    for row in rows:
        if row["days_kept"] > 14:
            total = total + (row["days_kept"] - 14)
    return total


everything = kit.sql("SELECT week, days_kept FROM issues ORDER BY issue_id")
by_python = {}
for week in (1, 2, 3, 4):
    rows = [r for r in everything if r["week"] == week]
    by_python[week] = (issues_in(rows), late_days(rows))

counts = {r["week"]: r["issues"] for r in kit.sql(
    "SELECT week, COUNT(*) AS issues FROM issues GROUP BY week ORDER BY week")}
lates = {r["week"]: r["late_days"] for r in kit.sql(
    "SELECT week, SUM(days_kept - 14) AS late_days FROM issues WHERE days_kept > 14 GROUP BY week ORDER BY week")}
by_sql = {week: (counts[week], lates[week]) for week in counts}

kit.table(["week", "issues", "late days"], [(w, *by_sql[w]) for w in sorted(by_sql)],
          caption="The librarian's weekly numbers, from two grouped queries")
kit.check("Python and SQL agree for all four weeks", by_python == by_sql, str(by_sql))

week,issues,late days
1,8,13
2,10,16
3,10,8
4,10,16


## Each Python move has a clause

The picture below is the one to carry into Week 2, which starts from these clauses.

In [14]:
kit.matrix(["the records", "keep some", "a running total", "one total per key", "sort"],
           ["in Python", "in SQL"],
           [["for issue in log", "FROM issues"],
            ["if ...", "WHERE ..."],
            ["count = count + 1", "COUNT(*), SUM(...)"],
            ["a dict keyed by dept", "GROUP BY dept"],
            ["sorted(..., reverse=True)", "ORDER BY ... DESC"]],
           title="The bridge from Python to SQL")
kit.check_summary()